# Step 01_05_04 -- Q4: Triple Survivorship Analysis (sc2egset)

**spec:** reports/specs/01_05_preregistration.md@7e259dd8
**Dataset:** sc2egset
**Branch:** feat/01-05-sc2egset
**Date:** 2026-04-18

**Objective:** Produce three survivorship tables:
1. Unconditional (§6.1): fraction_active per quarter
2. Sensitivity (§6.2): cohort sizes for N∈{5,10,20}
3. Conditional labels (§6.3): default N=10 flag

**Critique fixes:**
- B2: Primary PSI uncohort-filtered (T03). Survivorship here is SENSITIVITY.
- M5 fix: Hypothesis grain-explicit:
  (a) reference cohort N>=10 >= 50 players (falsifier < 50)
  (b) per-quarter active cohort >= 20 per quarter (falsifier: any < 20)

**Literature:**
- Gelman & Hill (2007) §12.5: min-cluster-size 10 obs/player for
  variance-stability in random-intercept fits.

In [ ]:
# spec: reports/specs/01_05_preregistration.md@7e259dd8
# Step 01_05_04 -- Q4 Survivorship triplet (sc2egset)
# Dataset: sc2egset  Branch: feat/01-05-sc2egset  Date: 2026-04-18

from pathlib import Path

import pandas as pd

from rts_predict.common.notebook_utils import get_notebook_db, get_reports_dir

In [ ]:
db = get_notebook_db("sc2", "sc2egset", read_only=True)
reports_dir = get_reports_dir("sc2", "sc2egset")
artifact_dir = reports_dir / "artifacts" / "01_exploration" / "05_temporal_panel_eda"
artifact_dir.mkdir(parents=True, exist_ok=True)

## Hypothesis (M5 fix: grain-explicit)

In [ ]:
# Hypothesis A: reference cohort (N>=10 matches in ref period 2022-08-29..2022-12-31)
# contains >= 50 distinct players.
# Falsifier A: cohort < 50 players -> PSI becomes anecdotal.
#
# Hypothesis B: fraction_active per quarter shows tournament cadence decay
# (some quarters empty/low due to no scheduled tournaments).
# Falsifier B: any quarter has zero active players in the overlap window.

## §6.1 Unconditional survivorship

In [ ]:
UNCOND_SQL = """
WITH players_in_window AS (
  SELECT DISTINCT player_id
  FROM matches_history_minimal
  WHERE started_at >= TIMESTAMP '2022-07-01'
    AND started_at <  TIMESTAMP '2025-01-01'
),
quarter_tagged AS (
  SELECT
    player_id,
    CAST(date_part('year', started_at) AS VARCHAR) || '-Q' ||
      CAST(date_part('quarter', started_at) AS VARCHAR) AS quarter
  FROM matches_history_minimal
  WHERE started_at >= TIMESTAMP '2022-07-01'
    AND started_at <  TIMESTAMP '2025-01-01'
    AND player_id IN (SELECT player_id FROM players_in_window)
)
SELECT
  quarter,
  COUNT(DISTINCT player_id) AS n_active,
  (SELECT COUNT(*) FROM players_in_window) AS n_total_in_window,
  COUNT(DISTINCT player_id) * 1.0 /
    (SELECT COUNT(*) FROM players_in_window) AS fraction_active
FROM quarter_tagged
GROUP BY quarter
ORDER BY quarter
"""

df_uncond = db.con.execute(UNCOND_SQL).fetchdf()
df_uncond["dataset_tag"] = "sc2egset"
print("Unconditional survivorship:")
print(df_uncond.to_string())

# Filter to overlap window
OVERLAP_QUARTERS = [
    "2022-Q3", "2022-Q4", "2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4",
    "2024-Q1", "2024-Q2", "2024-Q3", "2024-Q4",
]
df_uncond_overlap = df_uncond[df_uncond["quarter"].isin(OVERLAP_QUARTERS)].copy()
print(f"\nOverlap window ({len(df_uncond_overlap)} quarters):")
print(df_uncond_overlap.to_string())

## §6.2 Sensitivity cohort sizes for N∈{5,10,20}

In [ ]:
REF_COHORT_SQL = """
SELECT player_id, COUNT(*) AS n_ref_matches,
       MAX(started_at)::DATE - MIN(started_at)::DATE AS active_span_days
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2022-08-29'
  AND started_at <  TIMESTAMP '2023-01-01'
GROUP BY player_id
"""

df_ref_players = db.con.execute(REF_COHORT_SQL).fetchdf()
print(f"Reference period players: {len(df_ref_players)}")
print(df_ref_players["n_ref_matches"].describe())

# Compute cohort sizes for N∈{5,10,20} with active_span >= 30 days
sensitivity_rows = []
for n_threshold in [5, 10, 20]:
    cohort = df_ref_players[
        (df_ref_players["n_ref_matches"] >= n_threshold) &
        (df_ref_players["active_span_days"] >= 30)
    ]
    cohort_size = len(cohort)
    small_flag = "[SMALL-COHORT]" if cohort_size < 100 else ""
    print(f"N>={n_threshold} (span>=30d): {cohort_size} players {small_flag}")
    sensitivity_rows.append({
        "n_threshold": n_threshold,
        "cohort_size": cohort_size,
        "is_default": (n_threshold == 10),
        "small_cohort": cohort_size < 100,
        "notes": f"[SMALL-COHORT]" if cohort_size < 100 else "",
    })

df_sensitivity = pd.DataFrame(sensitivity_rows)
print("\nSensitivity table:")
print(df_sensitivity.to_string())

## Verdict

In [ ]:
# Hypothesis A check
cohort_n10 = df_sensitivity.loc[df_sensitivity["n_threshold"] == 10, "cohort_size"].iloc[0]
print(f"\nReference cohort N>=10: {cohort_n10} players")
if cohort_n10 >= 50:
    verdict_a = "CONFIRMED"
    print(f"# Verdict A: {verdict_a} — reference cohort has {cohort_n10} >= 50 players")
else:
    verdict_a = "FALSIFIED"
    print(f"# Verdict A: {verdict_a} — reference cohort has only {cohort_n10} < 50 players")
    print("  -> PSI conditional figures become anecdotal; uncohort-filtered PRIMARY (B2 fix) is justified.")

# Hypothesis B check
min_active = df_uncond_overlap["n_active"].min()
print(f"\nMin active players per quarter (overlap): {min_active}")
if min_active > 0:
    verdict_b = "CONFIRMED"
    print(f"# Verdict B: {verdict_b} — all overlap quarters have > 0 active players")
else:
    verdict_b = "FALSIFIED"
    print(f"# Verdict B: {verdict_b} — at least one quarter has zero active players")

# U4 resolution
print(f"\nU4 RESOLVED: cohort N=10: {cohort_n10} players")
if cohort_n10 < 20:
    print("  [SMALL-COHORT] Cross-dataset PSI conditional figure degrades to anecdote.")
elif cohort_n10 < 100:
    print("  [SMALL-COHORT] Cohort small but non-trivial.")

## Save artifacts

In [ ]:
out_uncond = artifact_dir / "survivorship_unconditional.csv"
df_uncond_overlap.to_csv(out_uncond, index=False)
print(f"Saved: {out_uncond}")

out_sens = artifact_dir / "survivorship_sensitivity.csv"
df_sensitivity.to_csv(out_sens, index=False)
print(f"Saved: {out_sens}")

# Verify
assert len(pd.read_csv(out_uncond)) == 10, "Expected 10 overlap quarters in survivorship_unconditional.csv"
assert len(pd.read_csv(out_sens)) >= 3, "Expected >= 3 rows in survivorship_sensitivity.csv"
print("Verification PASS")

## Markdown report

In [ ]:
md_content = f"""# Q4: Triple Survivorship Analysis — sc2egset

**spec:** reports/specs/01_05_preregistration.md@7e259dd8
**Date:** 2026-04-18

## SQL (verbatim, I6)

### Unconditional survivorship
```sql
{UNCOND_SQL.strip()}
```

### Reference cohort
```sql
{REF_COHORT_SQL.strip()}
```

## Unconditional survivorship (overlap window)

{df_uncond_overlap[['quarter','n_active','n_total_in_window','fraction_active','dataset_tag']].to_markdown(index=False)}

## Sensitivity cohort sizes (N∈{{5,10,20}}, active_span>=30d)

{df_sensitivity.to_markdown(index=False)}

## Verdicts

- Hypothesis A (reference cohort N>=10 >= 50): {verdict_a} (cohort={cohort_n10})
- Hypothesis B (all quarters non-empty): {verdict_b} (min_active={min_active})

## U4 resolution

Reference cohort N>=10 (span>=30d): **{cohort_n10} players**.
{("[SMALL-COHORT] Cohort small but non-trivial." if cohort_n10 < 100 else "Cohort size adequate.")}

## Notes

- B2 fix: Primary PSI in T03 is UNCOHORT-FILTERED. Survivorship sensitivity here
  informs the conditional-label captioning only.
- Gelman & Hill (2007) §12.5: min-cluster-size 10 obs/player for random-intercept fits.
- M5 fix: Hypothesis grain-explicit: (a) reference N>=50; (b) per-quarter n_active > 0.
"""

out_md = artifact_dir / "survivorship_sc2egset.md"
out_md.write_text(md_content)
print(f"Saved: {out_md}")

In [ ]:
db.close()
print("T05 complete.")